# Chapter 5: Pretraining on Unlabeled Data

This notebook covers pretraining a GPT model:
- Full training loop with loss tracking
- Learning rate scheduling
- Model checkpointing and evaluation
- Training on large text corpora

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import tiktoken
from tqdm import tqdm
import time
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 5.1: Prepare Training Data

In [ ]:
# Load tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Training text (in practice, use much larger datasets)
training_text = """
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. 
Large language models are trained on vast amounts of text data from the internet. These models learn to predict the next token in a sequence, 
which gives them the ability to generate coherent and contextually relevant text. Transformers use self-attention mechanisms to process sequences efficiently.
The transformer architecture has become the foundation for most modern NLP systems. Pretraining on unlabeled data helps models learn general language 
representations that can be fine-tuned for specific tasks. Fine-tuning involves training a pretrained model on task-specific labeled data.
Different pretraining objectives like masked language modeling and causal language modeling have different properties.
Natural language processing has many important applications including machine translation, question answering, and summarization.
Neural networks are inspired by biological neural systems and consist of interconnected layers of artificial neurons.
Deep learning has achieved state-of-the-art results in computer vision, natural language processing, and many other domains.
"" " * 5  # Repeat for more data

# Tokenize
tokens = tokenizer.encode(training_text)
print(f"Total tokens: {len(tokens)}")
print(f"Vocabulary size: {tokenizer.n_vocab}")

In [ ]:
# Create dataset
class PretrainingDataset(Dataset):
    def __init__(self, tokens, context_length):
        self.tokens = tokens
        self.context_length = context_length
    
    def __len__(self):
        return len(self.tokens) - self.context_length
    
    def __getitem__(self, idx):
        input_ids = self.tokens[idx:idx + self.context_length]
        target_ids = self.tokens[idx + 1:idx + self.context_length + 1]
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long)
        )

context_length = 128
dataset = PretrainingDataset(tokens, context_length)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(dataset, batch_size=32, shuffle=False, drop_last=True)

print(f"Dataset size: {len(dataset)}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 5.2: Learning Rate Scheduling

In [ ]:
class WarmupScheduler:
    """Learning rate scheduler with linear warmup and cosine annealing"""
    
    def __init__(self, optimizer, warmup_steps, total_steps, max_lr=1e-3, min_lr=1e-5):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        
        if self.current_step < self.warmup_steps:
            # Linear warmup
            lr = self.max_lr * (self.current_step / self.warmup_steps)
        else:
            # Cosine annealing
            progress = (self.current_step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr = self.min_lr + (self.max_lr - self.min_lr) * 0.5 * (1 + math.cos(math.pi * progress))
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
    
    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

print("WarmupScheduler implemented")

## 5.3: Training Function

In [ ]:
def train_epoch(model, train_loader, optimizer, scheduler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_tokens = 0
    
    pbar = tqdm(train_loader, desc="Training")
    for batch_idx, (input_ids, target_ids) in enumerate(pbar):
        input_ids, target_ids = input_ids.to(device), target_ids.to(device)
        
        # Forward pass
        logits = model(input_ids)
        
        # Reshape for loss
        batch_size, seq_len, vocab_size = logits.shape
        logits_flat = logits.reshape(-1, vocab_size)
        targets_flat = target_ids.reshape(-1)
        
        # Compute loss
        loss = nn.CrossEntropyLoss()(logits_flat, targets_flat)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        # Track metrics
        total_loss += loss.item() * batch_size
        total_tokens += batch_size * seq_len
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'lr': f'{scheduler.get_lr():.2e}'
        })
    
    avg_loss = total_loss / max(len(train_loader), 1) / batch_size
    return avg_loss

def evaluate(model, val_loader, device):
    """Evaluate model on validation set"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Evaluating")
        for input_ids, target_ids in pbar:
            input_ids, target_ids = input_ids.to(device), target_ids.to(device)
            
            logits = model(input_ids)
            batch_size, seq_len, vocab_size = logits.shape
            logits_flat = logits.reshape(-1, vocab_size)
            targets_flat = target_ids.reshape(-1)
            
            loss = nn.CrossEntropyLoss()(logits_flat, targets_flat)
            total_loss += loss.item()
    
    avg_loss = total_loss / len(val_loader)
    return avg_loss

print("Training and evaluation functions defined")

## 5.4: Full Training Loop

Note: This demonstrates the training procedure. For better results, use:
- Larger datasets
- More training steps
- Larger models
- Multi-GPU training

In [ ]:
# NOTE: This is a simplified example notebook cell
# In practice, you would import GPTModel from ch04_gpt_model
# For now, we'll show the training structure

print("""
Full Training Code Structure:

# Configuration
num_epochs = 10
warmup_steps = 100
total_steps = num_epochs * len(train_loader)
max_lr = 1e-3

# Initialize model
model = GPTModel(GPT_CONFIG, device).to(device)

# Optimizer and scheduler
optimizer = optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.01)
scheduler = WarmupScheduler(optimizer, warmup_steps, total_steps, max_lr)

# Training loop
best_val_loss = float('inf')

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss = evaluate(model, val_loader, device)
    
    print(f'Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}')
    
    # Checkpoint best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
""")

## 5.5: Distributed Training (DDP)

For scaling to multiple GPUs:

In [ ]:
# Example: Distributed Data Parallel training setup
print("""
Distributed Training with PyTorch DDP:

import torch.distributed as dist

# Initialize distributed training
dist.init_process_group(backend='nccl')
rank = dist.get_rank()
world_size = dist.get_world_size()

# Wrap model
model = GPTModel(config, device).to(rank)
model = nn.parallel.DistributedDataParallel(model, device_ids=[rank])

# Distributed sampler
train_sampler = DistributedSampler(
    dataset,
    num_replicas=world_size,
    rank=rank,
    shuffle=True
)

train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    sampler=train_sampler
)

# Run on multiple GPUs:
# torchrun --nproc_per_node=2 train.py
""")

## 5.6: Monitoring Training

Key metrics to track during pretraining:

In [ ]:
class TrainingMetrics:
    """Track training metrics"""
    
    def __init__(self):
        self.train_losses = []
        self.val_losses = []
        self.learning_rates = []
    
    def add_train_loss(self, loss):
        self.train_losses.append(loss)
    
    def add_val_loss(self, loss):
        self.val_losses.append(loss)
    
    def add_lr(self, lr):
        self.learning_rates.append(lr)
    
    def summary(self):
        if self.train_losses:
            print(f"Final train loss: {self.train_losses[-1]:.4f}")
            print(f"Final val loss: {self.val_losses[-1]:.4f}")
            print(f"Best val loss: {min(self.val_losses):.4f}")

print("TrainingMetrics class implemented")

## 5.7: Gradient Accumulation

For training with larger effective batch sizes:

In [ ]:
# Gradient accumulation example
print("""
Gradient Accumulation for Larger Effective Batch Sizes:

accumulation_steps = 4
actual_batch_size = 8
effective_batch_size = actual_batch_size * accumulation_steps  # 32

optimizer.zero_grad()

for step, (input_ids, target_ids) in enumerate(train_loader):
    logits = model(input_ids)
    loss = compute_loss(logits, target_ids)
    loss = loss / accumulation_steps  # Scale loss
    loss.backward()
    
    if (step + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
""")

## Summary

Key pretraining techniques:
1. **Data preparation**: Tokenize and create sliding windows
2. **Learning rate scheduling**: Warmup + cosine annealing
3. **Gradient clipping**: Stabilize training
4. **Checkpointing**: Save best model
5. **Distributed training**: Scale to multiple GPUs
6. **Gradient accumulation**: Increase effective batch size

With these techniques, you can pretrain powerful language models on large corpora!